In [1]:
import json
from utils import get_headers, get_api_domain
from urllib.parse import urljoin
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed


/Users/vasharma05/Projects/scripting/create-concept-set-members/my-py/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open('./fetch_all_concepts/all_uvl-labtests_concepts.json', 'r') as file:
  labtests = json.load(file)

with open('./fetch_all_concepts/all_uvl-procedures_concepts.json', 'r') as file:
  procedures = json.load(file)

with open('./fetch_all_concepts/all_uvl-forms_concepts.json', 'r') as file:
  forms = json.load(file)



In [3]:
fetched_cache = set()

def fetch_all_mappings(concept):
    if concept in fetched_cache:
        return []
    fetched_cache.add(concept)
    url = urljoin(urljoin(get_api_domain(), concept), "mappings/")
    params = {"limit": 100}
    mappings = []
    while url:
        response = requests.get(url, params=params, headers=get_headers())
        response.raise_for_status()
        data = response.json()
        mappings.extend(data)
        url = response.headers.get("next")
        params = {}
    mappings = list(
        filter(
            lambda mapping: mapping["map_type"] in {"CONCEPT-SET", "Q-AND-A"},
            mappings,
        )
    )
    ciel_mappings = list(
        filter(
            lambda mapping: mapping["to_source_owner"] == "CIEL",
            mappings
        ),
    )
    if len(ciel_mappings) > 0:
        print('TO BE CONVERTED', concept)
    return mappings

In [4]:
def get_all_concepts(concepts):
    all_concepts = set()
    all_concepts.update(concepts)
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = [executor.submit(fetch_all_mappings, concept) for concept in concepts]
        for future in as_completed(futures):
            mappings = future.result()
            mapped_concepts = list(
                map(lambda mapping: mapping["to_concept_url"], mappings)
            )
            mapped_concepts.extend(
                list(map(lambda mapping: mapping["from_concept_url"], mappings))
            )
            all_concepts.update(get_all_concepts(mapped_concepts))
    return all_concepts

In [5]:
fetched_cache = set()

all_labtests = get_all_concepts(list(map(lambda labtest: labtest['url'], labtests)))
print(len(all_labtests))

with open('./all_labtests.json', 'w') as f:
  json.dump(list(all_labtests), f)

all_procedures = get_all_concepts(list(map(lambda procedure: procedure['url'], procedures)))
print(len(all_procedures))

with open('./all_procedures.json', 'w') as f:
  json.dump(list(all_procedures), f)

all_forms = get_all_concepts(list(map(lambda form: form['url'], forms)))
print(len(all_forms))

with open('./all_forms.json', 'w') as f:
  json.dump(list(all_forms), f)


124
156
115
